<a href="https://colab.research.google.com/github/ns3271585-arch/Bayan/blob/main/notebooks/00_colab_setup.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# SDA-AIE-211 — Google Colab Setup
This notebook only prepares a GPU runtime for the **same GitHub repository** used locally. It contains no lab solutions.

Use it when a lab needs GPU training (primarily Labs 3–4). Do not use Colab GPU numbers as the Lab 7 CPU latency evidence.


## 1. Select a GPU runtime
In Colab choose **Runtime → Change runtime type → T4 GPU** (or another available GPU), then run the next cell.


In [ ]:
import sys, platform
import torch

print('Python:', sys.version.split()[0])
print('Platform:', platform.platform())
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only')
assert torch.cuda.is_available(), 'Enable a GPU runtime before Labs 3–4.'


## 2. Clone your own course repository
Replace `YOUR_GITHUB_USERNAME` with your GitHub username. For a private repository, authenticate using the method approved for your cohort; do not paste a token into a shared notebook.


In [ ]:
GITHUB_USERNAME = 'YOUR_GITHUB_USERNAME'
REPO_NAME = 'SDA-AIE-211-Bayan'

repo_url = f'https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git'
print(repo_url)
# Run once in a fresh runtime:
# !git clone $repo_url


In [ ]:
# After cloning, enter the project directory.
%cd /content/SDA-AIE-211-Bayan


## 3. Install the same project dependencies
The repository remains the source of truth. Colab installs from the same `requirements.txt` and editable package metadata.


In [ ]:
!python -m pip install --upgrade pip -q
!python -m pip install -r requirements.txt -q
!python -m pip install -e . -q


In [ ]:
!python scripts/doctor.py


## 4. Optional but recommended: persist large training artefacts in Google Drive
Colab runtimes are temporary. Mount Drive before long training so model artefacts survive a runtime reset. Code, tests, benchmark tables and decisions still belong in GitHub; large weights can live outside Git.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
from pathlib import Path

DRIVE_ARTIFACT_ROOT = Path('/content/drive/MyDrive/SDA-AIE-211/artifacts')
DRIVE_ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)
print('Persistent artefacts:', DRIVE_ARTIFACT_ROOT)


## 5. GPU lab pattern
When Labs 3–4 ask you to train, run the repository script rather than rewriting training logic in the notebook. The scripts expose output paths so the same code works locally or in Colab. Example pattern (do not run until that lab is implemented):

```bash
python scripts/train_classifier.py --output-dir /content/drive/MyDrive/SDA-AIE-211/artifacts/topic_classifier
```

After a run, commit/push **code + measured evidence**, not large model binaries.


## 6. Sync your code
At the beginning of a new Colab session: clone the repository again (or `git pull` if the runtime still exists). At the end of work: commit and push the code/evidence required by the lab.


# Lab 3 — Topic Classification, NER, and QA

This section contains the Google Colab GPU workflow and results for Lab 3.

- Topic Classification: TF-IDF baseline and XLM-R classifier
- Named Entity Recognition (NER): XLM-R token classification
- Question Answering (QA): extractive QA smoke test

In [1]:
import os
import sys
import torch

REPO = "/content/Bayan"

# Clone the repository only if it is not already available
if not os.path.exists(os.path.join(REPO, ".git")):
    !git clone https://github.com/ns3271585-arch/Bayan.git /content/Bayan
else:
    %cd /content/Bayan
    !git pull origin main

%cd /content/Bayan

# Allow Python to import the Bayan package directly from src
os.environ["PYTHONPATH"] = "/content/Bayan/src"
if "/content/Bayan/src" not in sys.path:
    sys.path.insert(0, "/content/Bayan/src")

print("Python:", sys.version.split()[0])
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")
print("Repository:", os.getcwd())

Cloning into '/content/Bayan'...
remote: Enumerating objects: 158, done.
remote: Counting objects: 100% (158/158), done.
remote: Compressing objects: 100% (124/124), done.
remote: Total 158 (delta 41), reused 140 (delta 24), pack-reused 0 (from 0)
Receiving objects: 100% (158/158), 656.92 KiB | 7.91 MiB/s, done.
Resolving deltas: 100% (41/41), done.
/content/Bayan
Python: 3.13.15
CUDA available: True
GPU: Tesla T4
Repository: /content/Bayan


In [2]:
!pip install -q transformers==5.13.1 datasets accelerate evaluate scikit-learn sentencepiece seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.5/11.5 MB 107.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 109.3 MB/s eta 0:00:00


In [3]:
import transformers
print("transformers version:", transformers.__version__)

transformers version: 5.13.1


In [4]:
!pytest tests/test_model_data.py tests/test_ner_alignment.py tests/test_qa.py -q

...........                                                              [100%]
11 passed in 1.43s


In [5]:
!python scripts/tfidf_baseline.py

Grouped dataset sizes:
Train:      8389
Validation: 2403
Test:       1208

Unique citizens:
Train:      3360
Validation: 960
Test:       480

TF-IDF vocabulary size: 1,037

VALIDATION
Macro-F1: 1.0000
Accuracy: 1.0000

                  precision    recall  f1-score   support

         billing     1.0000    1.0000    1.0000       304
digital_services     1.0000    1.0000    1.0000       294
       licensing     1.0000    1.0000    1.0000       289
        lighting     1.0000    1.0000    1.0000       313
           parks     1.0000    1.0000    1.0000       329
           roads     1.0000    1.0000    1.0000       289
           waste     1.0000    1.0000    1.0000       316
           water     1.0000    1.0000    1.0000       269

        accuracy                         1.0000      2403
       macro avg     1.0000    1.0000    1.0000      2403
    weighted avg     1.0000    1.0000    1.0000      2403

FROZEN TEST
Macro-F1: 1.0000
Accuracy: 1.0000

                  precision    reca

In [6]:
!python scripts/train_classifier.py

BAYAN LAB 3A — XLM-R TOPIC CLASSIFIER

Dataset sizes:
Train:      8389
Validation: 2403
Test:       1208

Labels:
{'billing': 0, 'digital_services': 1, 'licensing': 2, 'lighting': 3, 'parks': 4, 'roads': 5, 'waste': 6, 'water': 7}

Loading tokenizer: xlm-roberta-base
config.json: 100% 615/615 [00:00<00:00, 2.65MB/s]
tokenizer_config.json: 100% 25.0/25.0 [00:00<00:00, 124kB/s]
sentencepiece.bpe.model: 100% 5.07M/5.07M [00:00<00:00, 18.4MB/s]
tokenizer.json: 100% 9.10M/9.10M [00:00<00:00, 64.7MB/s]
Map: 100% 8389/8389 [00:00<00:00, 20575.66 examples/s]
Map: 100% 2403/2403 [00:00<00:00, 21590.48 examples/s]
Map: 100% 1208/1208 [00:00<00:00, 20536.56 examples/s]

Loading model: xlm-roberta-base

model.safetensors: downloading bytes:  16% 182M/1.12G [00:01<00:03, 253MB/s, 14.5MB/s  ]
model.safetensors: downloading bytes:  20% 221M/1.12G [00:01<00:03, 225MB/s, 19.7MB/s  ]
model.safetensors: downloading bytes:  36% 398M/1.12G [00:01<00:02, 286MB/s, 33.7MB/s  ]
model.safetensors: downloading b

In [7]:
!python scripts/train_ner.py

BAYAN LAB 3B — XLM-R NER

Total sentences: 4000
Total tokens: 36000

Labels:
{'O': 0, 'B-SERVICE': 1, 'B-LOCATION': 2, 'B-DATE': 3, 'B-REFERENCE': 4}

Dataset sizes:
Train:      3200
Validation: 400
Test:       400

Loading tokenizer: xlm-roberta-base
Map: 100% 3200/3200 [00:00<00:00, 9565.16 examples/s]
Map: 100% 400/400 [00:00<00:00, 10340.22 examples/s]
Map: 100% 400/400 [00:00<00:00, 6223.79 examples/s]

Loading model: xlm-roberta-base
Loading weights: 100% 197/197 [00:00<00:00, 679.17it/s]
[transformers] XLMRobertaForTokenClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
classifier.weight    

In [8]:
!python scripts/qa_smoke.py

BAYAN LAB 3B — QA SMOKE TEST
Examples:   12
Answerable: 9
Null:       3

Device: cuda
Checkpoint: deepset/roberta-base-squad2

config.json: 100% 571/571 [00:00<00:00, 2.85MB/s]
tokenizer_config.json: 100% 79.0/79.0 [00:00<00:00, 367kB/s]
vocab.json: 100% 899k/899k [00:00<00:00, 26.1MB/s]
merges.txt: 100% 456k/456k [00:00<00:00, 53.7MB/s]
special_tokens_map.json: 100% 772/772 [00:00<00:00, 3.30MB/s]

model.safetensors: downloading bytes:  41% 203M/496M [00:01<00:01, 251MB/s, 15.3MB/s  ]
model.safetensors: downloading bytes:  94% 464M/496M [00:02<00:00, 205MB/s, 40.0MB/s  ]
model.safetensors: reconstructing file:  52% 256M/496M [00:05<00:05, 44.6MB/s, 6.46MB/s  ]
model.safetensors: downloading bytes: 100% 472M/472M [00:09<00:00, 48.0MB/s, 41.3MB/s  ]
model.safetensors: reconstructing file: 100% 496M/496M [00:09<00:00, 50.5MB/s, 34.6MB/s  ]
Loading weights: 100% 199/199 [00:00<00:00, 4222.85it/s]
QA-0001: predicted='2 business days' expected=['2 business days'] PASS
QA-0002: predicted='50

## Lab 3 Results Summary

### Topic Classification — TF-IDF Baseline
- Validation macro-F1: **1.0000**
- Frozen test macro-F1: **1.0000**
- Validation accuracy: **1.0000**
- Frozen test accuracy: **1.0000**

### Topic Classification — XLM-R
- Validation macro-F1: **1.0000**
- Frozen test macro-F1: **0.9992**
- Validation accuracy: **1.0000**
- Frozen test accuracy: **0.9992**

### Named Entity Recognition — XLM-R
- Validation entity-F1: **1.0000**
- Frozen test entity-F1: **1.0000**
- Validation precision: **1.0000**
- Validation recall: **1.0000**
- Frozen test accuracy: **1.0000**

### Extractive Question Answering
- Answerable questions: **9/9**
- Unanswerable (null) questions: **3/3**
- Overall: **12/12**
- QA smoke target: **PASS**

### Baseline Ceiling Note
The TF-IDF baseline achieved a macro-F1 of 1.0000 on this dataset. Therefore, a requirement of improving the transformer model by +0.08 over the baseline is mathematically unattainable, since macro-F1 cannot exceed 1.0000. The actual model results are reported without artificially weakening the baseline.